# MLP para reconocimiento de paisajes

Proyecto de **Machine Learning** en el que se entrena un **Perceptrón Multicapa (MLP)**, un tipo de red neuronal, para clasificar imágenes de paisajes en 6 categorías. Se usa `TensorFlow`/`Keras` para cargar y procesar las imágenes, y `Matplotlib`/`Seaborn` para graficar los resultados.

## 2. Problema de negocio y propuesta

### a. Contexto del problema

Se necesita un modelo de inteligencia artificial capaz de recibir una fotografía de un paisaje y decir a qué categoría pertenece entre 6 opciones: edificios, bosques, glaciares, montañas, mar y calles. Este tipo de problema se llama **clasificación de imágenes**: el modelo no "ve" como nosotros, sino que recibe una matriz de números (los píxeles) y debe aprender qué números suelen aparecer en cada paisaje.

Las imágenes originales miden **150x150 píxeles y tienen 3 canales de color (RGB)**, es decir, 150×150×3 = **67.500 números** por foto. La red debe resumir toda esa información para decidir entre 6 respuestas posibles.

### b. Importancia para el negocio

Clasificar paisajes de forma automática tiene valor en varios rubros:

*   **Turismo:** recomendar destinos según el tipo de paisaje de una foto.
*   **Medio ambiente:** monitorear bosques, glaciares o el mar de forma automática.
*   **Redes sociales:** etiquetar fotos para poder buscarlas mejor.
*   **Ciudades inteligentes:** distinguir zonas urbanas (calles, edificios) de zonas naturales.

Hacerlo de forma manual con miles de fotos es lento y costoso. Un modelo entrenado puede clasificar las imágenes en segundos, por eso la propuesta de este notebook es construir ese modelo y medir con métricas qué tan bien lo hace.

## 3. Selección y descripción del dataset

### a. Dataset utilizado

Se utilizó el dataset **Intel Image Classification**, un conjunto de imágenes de paisajes naturales y urbanos muy usado para aprender Machine Learning. Viene organizado en tres carpetas según su propósito:

*   `seg_train`: imágenes **etiquetadas** para entrenar al modelo (tienen subcarpetas por clase).
*   `seg_test`: imágenes **etiquetadas** para la prueba final ("examen") del modelo.
*   `seg_pred`: imágenes **sin etiqueta** que servirían para predicciones sobre fotos nuevas.

### b. Cantidad de imágenes

En total hay aproximadamente **25.000 imágenes**, repartidas así:

*   `seg_train`: **14.034** imágenes.
*   `seg_test`: **3.000** imágenes.
*   `seg_pred`: **7.301** imágenes.

*Las cantidades exactas se verifican con código en el análisis exploratorio.*

### c. Número de clases

El dataset tiene **6 clases** (categorías), que son los 6 tipos de paisaje que el modelo debe distinguir:

1.  `buildings` (edificios)
2.  `forest` (bosque)
3.  `glacier` (glaciar)
4.  `mountain` (montaña)
5.  `sea` (mar)
6.  `street` (calle)

Como hay 6 categorías, la capa de salida del modelo tendrá 6 neuronas.

### d. Características principales

*   **Formato:** imágenes `.jpg`.
*   **Resolución original:** 150x150 píxeles (se confirma con código más adelante).
*   **Canales:** 3 canales de color (RGB).
*   **Organización:** una subcarpeta por clase dentro de `seg_train` y `seg_test`.
*   **Etiquetado:** el nombre de cada subcarpeta ES la clase de las fotos que contiene.

### e. Justificación de selección

Elegimos este dataset porque:

*   Es **variado**: mezcla zonas naturales y urbanas, por lo que el modelo debe aprender patrones distintos (colores, formas y texturas).
*   Está **bien etiquetado**: las carpetas ya llevan el nombre de la clase, no hay que etiquetar a mano.
*   Es de **tamaño razonable**: ~25.000 imágenes permiten aprender sin necesitar un supercomputador.
*   Permite recorrer el **ciclo completo** de un proyecto de ML: analizar, preprocesar, entrenar y evaluar.

## 4. Análisis exploratorio de imágenes (EDA)

Antes de entrenar, conviene *conocer* los datos: cuántas fotos hay por clase, qué resolución tienen y qué tan parecidas o diferentes son. Eso es exactamente lo que se hace en un análisis exploratorio.

### a. Distribución de clases

¿Hay la misma cantidad de fotos en cada clase? Si una clase tuviera muchísimas más fotos, el modelo podría "copiar" la clase más común y olvidarse del resto. Lo verificamos con código.

In [ ]:
import os

# Ruta a la carpeta ORIGINAL (no la modificamos)
ruta_original = os.path.join("dataset", "Intel Image Classification")
ruta_train_original = os.path.join(ruta_original, "seg_train", "seg_train")
ruta_test_original = os.path.join(ruta_original, "seg_test", "seg_test")

# Listamos las 6 clases
clases = sorted(os.listdir(ruta_train_original))
print("Clases encontradas:", clases, "\n")

# Contamos cuántas fotos hay por clase en train y en test
conteo_train = {}
conteo_test = {}
for clase in clases:
    conteo_train[clase] = len(os.listdir(os.path.join(ruta_train_original, clase)))
    conteo_test[clase] = len(os.listdir(os.path.join(ruta_test_original, clase)))
    print(f"{clase:10s} -> train: {conteo_train[clase]:5d} | test: {conteo_test[clase]:5d}")

print("\nTotales:")
print(f"  Train: {sum(conteo_train.values())}")
print(f"  Test:  {sum(conteo_test.values())}")
print(f"  Pred:  {len(os.listdir(os.path.join(ruta_original, 'seg_pred', 'seg_pred')))}")

**Resultado:** las 6 clases tienen una cantidad **muy parecida** de imágenes (entre ~2.200 y ~2.500 en train, y ~450–550 en test). No hay un desbalanceo importante, por lo que **no hace falta balancear** las clases.

### b. Resolución y canales

Aquí verificamos el **tamaño real** de las imágenes y que sean a color. Usamos el cargador de TensorFlow `image_dataset_from_directory`, que convierte cada subcarpeta en lotes de tensores. Cargamos **sin redimensionar** para ver la resolución original del dataset.

In [ ]:
import tensorflow as tf

# Cargamos un lote SIN redimensionar para ver la resolución real de las fotos
ds_prueba_resolucion = tf.keras.utils.image_dataset_from_directory(
    ruta_test_original,
    batch_size=32
)

for imagenes, _ in ds_prueba_resolucion.take(1):
    print("Forma de un lote completo:", imagenes.shape)
    print("Forma de una sola imagen:", imagenes[0].shape)
    print("Tipo de datos (dtype):", imagenes.dtype)
    break

**Resultado:** cada lote tiene forma `(32, 150, 150, 3)`: 32 imágenes de **150x150** píxeles con **3 canales** (RGB, o sea a color). El `dtype` es `uint8`, lo que significa valores de píxel entre **0 y 255**.

### c. Patrones visuales

Al observar las fotos de cada clase se distinguen patrones típicos:

*   **buildings**: predominan líneas rectas y formas rectangulares (ventanas, paredes).
*   **forest**: domina el color verde en distintas tonalidades, con troncos o ramas.
*   **glacier**: blancos, azules y celestes, con formas más curvas.
*   **mountain**: se ven picos o formas triangulares.
*   **sea**: distintas tonalidades de azul y pocas figuras definidas.
*   **street**: espacios angostos con objetos urbanos y, a veces, personas.

### d. Variabilidad encontrada

Dentro de una misma clase hay variabilidad: distintas horas de luz, ángulos, estaciones y cercanía. Por ejemplo, algunas fotos de `glacier` se parecen a `sea` por el color azul, y algunas de `mountain` pueden parecerse a `forest` si hay muchos árboles. Esa variabilidad es normal y será uno de los grandes retos del modelo.

## 5. Preprocesamiento de datos

### a. Carga y limpieza

Para no dañar el dataset original, primero creamos una **copia de seguridad** con `shutil`. Todo el preprocesamiento se hará sobre la **copia**, así la fuente original queda intacta y siempre podemos volver a ella.

In [ ]:
import os
import shutil

ruta_origen = os.path.join("dataset", "Intel Image Classification")
ruta_copia = os.path.join("copy", "Intel Image Classification")

if os.path.exists(ruta_copia):
    print("La copia ya existe. La reutilizamos para no duplicar el trabajo.")
else:
    print("Creando copia del dataset (esto puede tardar unos segundos)...")
    os.makedirs("copy", exist_ok=True)
    shutil.copytree(ruta_origen, ruta_copia)
    print("Copia creada correctamente.")

A partir de aquí, las rutas apuntan a la **copia** y no a la fuente original.

In [ ]:
# Redirigimos las rutas de trabajo hacia la copia
ruta_train = os.path.join(ruta_copia, "seg_train", "seg_train")
ruta_test = os.path.join(ruta_copia, "seg_test", "seg_test")

print("Ruta train (copia):", ruta_train)
print("Ruta test  (copia):", ruta_test)

Ahora limpiamos la copia: recorremos todos los archivos y nos quedamos solo con imágenes (extensiones `.jpg`, `.jpeg` y `.png`). Cualquier otro archivo se elimina para que el cargador no falle.

In [ ]:
import os

extensiones_validas = {'.jpg', '.jpeg', '.png'}
archivos_eliminados = 0

for ruta_carpeta in [ruta_train, ruta_test]:
    for clase in os.listdir(ruta_carpeta):
        ruta_clase = os.path.join(ruta_carpeta, clase)
        if not os.path.isdir(ruta_clase):
            continue
        for archivo in os.listdir(ruta_clase):
            _, extension = os.path.splitext(archivo)
            if extension.lower() not in extensiones_validas:
                os.remove(os.path.join(ruta_clase, archivo))
                print("Archivo inválido eliminado:", archivo)
                archivos_eliminados += 1

print(f"Limpieza finalizada. Archivos inválidos eliminados: {archivos_eliminados}")

### b. Redimensionamiento

Las fotos originales miden **150x150**, pero para un MLP reducir el tamaño es una gran ventaja: las reducimos a **64x64** con `image_size=(64, 64)`.

*   Con 150x150, la entrada tendría 150×150×3 = **67.500 valores** y la primera capa oculta ~34,5 millones de parámetros: demasiado pesada y muy propensa a memorizar (overfitting).
*   Con 64x64 quedan 64×64×3 = **12.288 valores** y ~6,3 millones de parámetros: entrena más rápido, ocupa menos memoria (importante en esta PC) y sobreajusta menos.

Además, el redimensionamiento garantiza que todas las fotos entren a la red con **exactamente la misma forma**, tal como la red lo exige.

In [ ]:
import tensorflow as tf

# Demostración: cargamos la copia ya redimensionada a 64x64
ds_demo = tf.keras.utils.image_dataset_from_directory(
    ruta_test,
    image_size=(64, 64),
    batch_size=32
)

for imagenes, _ in ds_demo.take(1):
    print("Forma del lote redimensionado:", imagenes.shape)
    break

### c. Normalización

Los píxeles van de 0 a 255, pero las redes aprenden **mucho mejor con números pequeños y parecidos**. Por eso dividimos cada píxel entre 255 con `Rescaling(1/255)`: todos los valores quedan entre **0 y 1**.

In [ ]:
# Capa que normaliza: divide cada píxel entre 255 (máximo valor de RGB)
normalizador = tf.keras.layers.Rescaling(1. / 255)

for imagenes, _ in ds_demo.take(1):
    pixel_original = imagenes[0][0][0].numpy()
    pixel_normalizado = normalizador(imagenes)[0][0][0].numpy()
    print("Primer píxel SIN normalizar:", pixel_original)
    print("Primer píxel  normalizado  :", pixel_normalizado)
    break

**Resultado:** el píxel pasó de un valor entero entre 0–255 a un **decimal entre 0 y 1**. Con estos números la red aprende de forma más estable y rápida.

### d. Etiquetado

Keras asigna automáticamente un **número entero (etiqueta)** a cada clase, usando el nombre de las subcarpetas en orden alfabético:

*   0 -> `buildings`
*   1 -> `forest`
*   2 -> `glacier`
*   3 -> `mountain`
*   4 -> `sea`
*   5 -> `street`

El modelo no devuelve "forest": devuelve **6 probabilidades**, y nosotros elegimos la clase con la probabilidad más alta.

In [ ]:
# Keras guarda el nombre de cada clase en .class_names
print("Etiqueta (entero) -> Clase")
for indice, nombre in enumerate(ds_demo.class_names):
    print(f"  {indice} -> {nombre}")

for _, etiquetas in ds_demo.take(1):
    print("\nMuestra de etiquetas de un lote:", etiquetas.numpy()[:10])
    break

### e. Train / Validation / Test

Dividimos los datos en 3 grupos para que la evaluación sea **honesta**:

*   **Train (80% de seg_train):** fotos con las que el modelo estudia y ajusta sus pesos.
*   **Validation (20% de seg_train):** fotos ocultas que el modelo usa como "ensayo" al final de cada época, **sin ajustar sus pesos** con ellas.
*   **Test (100% de seg_test):** el "examen final", fotos que el modelo **nunca** vio durante el entrenamiento.

Usamos la misma semilla (`seed=123`) en train y validation para que la división sea idéntica en ambas llamadas y no se mezclen las fotos.

In [ ]:
import tensorflow as tf

SEMILLA = 123          # fija la división aleatoria (reproducibilidad)
TAMANO = (64, 64)      # redimensionamiento a 64x64
BATCH = 32             # lotes de 32 imágenes (protege la RAM)

# 80% de seg_train para entrenar
dataset_train = tf.keras.utils.image_dataset_from_directory(
    ruta_train,
    validation_split=0.2,
    subset="training",
    seed=SEMILLA,
    image_size=TAMANO,
    batch_size=BATCH
)

# 20% de seg_train para validar
dataset_val = tf.keras.utils.image_dataset_from_directory(
    ruta_train,
    validation_split=0.2,
    subset="validation",
    seed=SEMILLA,
    image_size=TAMANO,
    batch_size=BATCH
)

# 100% de seg_test para el examen final
dataset_test = tf.keras.utils.image_dataset_from_directory(
    ruta_test,
    image_size=TAMANO,
    batch_size=BATCH
)

print("\nConjuntos listos:")
print("  Train:", len(dataset_train), "lotes")
print("  Validación:", len(dataset_val), "lotes")
print("  Test:", len(dataset_test), "lotes")
print("  Clases:", dataset_train.class_names)

Se cargaron los tres conjuntos **por lotes de 32 imágenes** (`batch_size=32`): en memoria nunca está todo el dataset de golpe, lo que **protege la RAM** de la PC.

Sobre estos lotes aplicamos dos transformaciones a los datos de **entrenamiento**:

1.  **Normalización** con `Rescaling`: píxeles entre 0 y 1.
2.  **Aumento de datos**: voltear horizontalmente, rotar levemente y hacer zoom pequeño, de forma aleatoria en cada época. Así la red nunca ve dos veces exactamente la misma foto y aprende a **generalizar** (reduce el overfitting).

La **validación y el test** solo se normalizan, para evaluar con las fotos reales, sin alteraciones.

In [ ]:
normalizador = tf.keras.layers.Rescaling(1. / 255)

# Aumento de datos: SOLO se aplica al entrenamiento
aumento_datos = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),  # volteo horizontal aleatorio
    tf.keras.layers.RandomRotation(0.05),      # rotación leve aleatoria
    tf.keras.layers.RandomZoom(0.05),          # zoom leve aleatorio
], name="Aumento_de_datos")

def preparar_entrenamiento(imagen, etiqueta):
    imagen = normalizador(imagen)   # píxeles a [0,1]
    imagen = aumento_datos(imagen)  # variación aleatoria
    return imagen, etiqueta

def preparar_evaluacion(imagen, etiqueta):
    imagen = normalizador(imagen)   # solo normalizar
    return imagen, etiqueta

dataset_train_prep = dataset_train.map(
    preparar_entrenamiento, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
dataset_val_prep = dataset_val.map(
    preparar_evaluacion, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)
dataset_test_prep = dataset_test.map(
    preparar_evaluacion, num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

print("Pipeline listo: train con normalización + aumento, val/test solo con normalización.")

In [ ]:
# Verificamos que los datos de entrenamiento ya estén normalizados (0 a 1)
for imagenes, etiquetas in dataset_train_prep.take(1):
    print("Forma del lote:", imagenes.shape)
    print("Primeras etiquetas:", etiquetas.numpy()[:8])
    print("Primer píxel (normalizado):", imagenes[0][0][0].numpy())
    break

### f. Justificación de las transformaciones

*   **Copia y limpieza:** protegen la fuente original y evitan errores por archivos basura.
*   **Redimensionamiento a 64x64:** reduce el número de entradas de 67.500 a 12.288, con menos parámetros el MLP entrena más rápido, ocupa menos RAM y sobreajusta menos.
*   **Normalización:** pone todos los valores en la misma escala (0-1), lo que hace el aprendizaje más rápido y estable.
*   **Etiquetado automático:** Keras convierte el nombre de la subcarpeta en un entero, sin trabajo manual.
*   **División train/val/test:** nos permite medir aprendizaje real y no memoria.
*   **Aumento de datos:** crea variaciones de cada foto para que el modelo **generalice** mejor y no se sobreajuste.

## 6. Implementación del MLP

### a. Arquitectura

Un **MLP** (Perceptrón Multicapa) es una red donde cada neurona de una capa se conecta con **todas** las neuronas de la siguiente; por eso también se llama *fully connected* o *feed forward*. Como las imágenes llegan como matrices (64x64x3), el primer paso es **aplanarlas** con una capa `Flatten` para convertirlas en un vector de **12.288** números.

### b. Capas y neuronas

La arquitectura final del modelo:

*   **Entrada (`Flatten`):** convierte 64x64x3 = 12.288 valores en un vector.
*   **Capa oculta 1:** **512 neuronas** con activación `ReLU`.
*   **`BatchNormalization`:** re-escala y estabiliza la activación de la capa anterior (evita que la red "muera" con ReLU y acelera el aprendizaje).
*   **`Dropout` 50%:** apaga la mitad de las neuronas al azar en cada pasada.
*   **Capa oculta 2:** **128 neuronas** con activación `ReLU`.
*   **`BatchNormalization`:** estabiliza la segunda capa oculta.
*   **`Dropout` 40%:** apaga el 40% de las neuronas al azar.
*   **Salida:** **6 neuronas** con `Softmax` (una por cada clase).

La forma de "embudo" (512 → 128) captura primero muchos patrones visuales y luego los resume en características más abstractas antes de decidir.

### c. Funciones de activación

Las funciones de activación deciden si una neurona se "enciende" o no:

*   **ReLU** (capas ocultas): es simple y rápida. Si la entrada es negativa devuelve 0; si es positiva la deja pasar igual. Evita que el modelo aprenda demasiado lento.
*   **Softmax** (salida): convierte los 6 puntajes en **probabilidades que suman 1**, para leer la respuesta como un porcentaje de confianza por clase.

> **Idea de primer año:** ReLU es como un semáforo que solo deja pasar números positivos; Softmax reparte un "pastel" de probabilidad entre las 6 clases, y la clase con la tajada más grande es la predicción.

### d. Función de pérdida

La **función de pérdida (loss)** mide qué tan equivocado está el modelo y el **optimizador** usa ese error para ajustar los pesos.

*   **`sparse_categorical_crossentropy`:** ideal para clasificar en varias clases cuando la etiqueta es un entero (0 a 5), sin necesidad de codificar one-hot.
*   **`Adam`:** optimizador de **descenso de gradiente** con "memoria". Le bajamos la **tasa de aprendizaje a 0.0005** (menor que el valor por defecto de 0.001) para que avance con pasos más cortos y **no salte el mejor valor**.

> **Idea de primer año:** la pérdida baja cuando el modelo mejora. El descenso de gradiente es como bajar una montaña con niebla: si das pasos enormes te pasas del valle, por eso aquí damos pasos más cortos.

### e. Justificación de las decisiones

*   **64x64 en la entrada:** baja de 67.500 a 12.288 características: menos parámetros, menos memoria y menos overfitting.
*   **512 y 128 neuronas:** capacidad suficiente para aprender patrones de paisajes sin hacer la red excesivamente profunda (evita el desvanecimiento del gradiente).
*   **`BatchNormalization` después de cada capa oculta:** el primer intento con dropout sobre una entrada enorme **colapsaba** (la red predecía siempre la misma clase). La normalización por lotes estabiliza los valores que recibe cada neurona y evita ese colapso.
*   **Dropout alto (50% y 40%):** apagar neuronas al azar obliga a la red a no depender de unas pocas neuronas "estrella" y a generalizar mejor.
*   **Tasa de aprendizaje 0.0005:** menor que la por defecto, para converger de forma fina y estable.
*   **ReLU + Softmax:** activaciones estándar y eficientes para clasificación multiclase.
*   **Máximo 30 épocas con vigilancia:** si el error de validación deja de mejorar, el entrenamiento se detiene y se recuperan los mejores pesos (no se sobreentrena por inercia).

In [ ]:
import tensorflow as tf

modelo = tf.keras.Sequential([
    # Entrada: aplana 64x64x3 en un vector de 12.288 números
    tf.keras.layers.Flatten(input_shape=(64, 64, 3), name="Aplanar_entrada"),

    # Capa oculta 1: 512 neuronas con ReLU
    tf.keras.layers.Dense(512, activation='relu', name="Capa_oculta_1"),
    tf.keras.layers.BatchNormalization(name="BatchNorm_1"),

    # Dropout 50%: apaga la mitad de las neuronas al azar
    tf.keras.layers.Dropout(0.5, name="Dropout_1"),

    # Capa oculta 2: 128 neuronas con ReLU
    tf.keras.layers.Dense(128, activation='relu', name="Capa_oculta_2"),
    tf.keras.layers.BatchNormalization(name="BatchNorm_2"),

    # Dropout 40%: apaga el 40% de las neuronas al azar
    tf.keras.layers.Dropout(0.4, name="Dropout_2"),

    # Salida: 6 neuronas con Softmax (una probabilidad por clase)
    tf.keras.layers.Dense(6, activation='softmax', name="Salida")
])

# Tabla resumen de capas, formas y cantidad de parámetros
modelo.summary()

# Compilamos con un descenso de gradiente MÁS LENTO que el por defecto (lr=0.0005)
modelo.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)
print("\nModelo compilado: Adam lr=0.0005 + sparse_categorical_crossentropy")

**Leyendo el `summary`:** la columna `Output Shape` muestra el tamaño que sale de cada capa (el `None` es el lote, que Keras deja abierto). La columna `Param #` indica los pesos y sesgos de la capa: la entrada aplanada no tiene pesos (0), la primera capa oculta (512 neuronas con entrada de 12.288 valores) tiene ~6,3 millones de parámetros y la capa de salida solo 774. En total el modelo tiene ~**6,36 millones** de parámetros (muy por debajo de los ~34,6 millones que tendría con la entrada de 150x150). `BatchNormalization` añade pocos parámetros extra, y `Dropout` no tiene parámetros (solo apaga neuronas).

## 7. Entrenamiento y validación

### a. Métricas de entrenamiento y validación

Durante cada época se imprimen 4 valores:

*   **`loss` (train):** qué tan equivocado está con las fotos de estudio.
*   **`accuracy` (train):** porcentaje de aciertos con las fotos de estudio.
*   **`val_loss` (validación):** qué tan equivocado está con el 20% oculto.
*   **`val_accuracy` (validación):** porcentaje de aciertos con el 20% oculto (el "examen sorpresa").

Configuramos dos **vigilantes** (callbacks) para cuidar el entrenamiento:

*   **`EarlyStopping`:** vigila `val_loss`; si no mejora durante 5 épocas, detiene el entrenamiento y restaura los mejores pesos.
*   **`ReduceLROnPlateau`:** si `val_loss` no mejora durante 3 épocas seguidas, **le baja la velocidad al descenso de gradiente** (reduce la tasa a la mitad) hasta un mínimo de 1e-6.

In [ ]:
# Vigilante 1: detiene el entrenamiento si la validación deja de mejorar
parada_temprana = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

# Vigilante 2: baja la velocidad del gradiente cuando se estanca
reducir_velocidad = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-6
)

# Entrenamiento con el modelo con Dropout (máximo 30 épocas)
historial = modelo.fit(
    dataset_train_prep,                # fotos de estudio (train)
    validation_data=dataset_val_prep,  # examen sorpresa (validation)
    epochs=30,                         # 30 pasadas completas como máximo
    callbacks=[parada_temprana, reducir_velocidad]
)

**Leyendo los valores:** el `loss` y el `accuracy` muestran el desempeño con el entrenamiento; el `val_loss` y el `val_accuracy` muestran la calificación del "ensayo final" con el 20% de fotos ocultas. Como la validación no ajusta pesos, un `val_accuracy` alto indica que el modelo **entendió** el concepto y no solo se memorizó las fotos de estudio. En un modelo con Dropout es normal que el `accuracy` de entrenamiento sea un poco menor que el de validación, porque las neuronas se "apagan" al azar mientras estudia.

### b. Curvas de Accuracy y de Loss

Grafiquemos cómo evolucionaron la exactitud y la pérdida durante el entrenamiento, tanto en train como en validation. De estas curvas se detecta si el modelo aprendió de verdad o solo memorizó.

In [ ]:
import matplotlib.pyplot as plt

acc = historial.history['accuracy']
val_acc = historial.history['val_accuracy']
loss = historial.history['loss']
val_loss = historial.history['val_loss']

epocas = range(1, len(acc) + 1)

plt.figure(figsize=(14, 5))

# Gráfico 1: Accuracy
plt.subplot(1, 2, 1)
plt.plot(epocas, acc, 'b-', label='Entrenamiento (Accuracy)')
plt.plot(epocas, val_acc, 'r-', label='Validación (Val_Accuracy)')
plt.title('Evolución de la Exactitud (Accuracy)')
plt.xlabel('Épocas')
plt.ylabel('Exactitud')
plt.legend()
plt.grid(True)

# Gráfico 2: Loss
plt.subplot(1, 2, 2)
plt.plot(epocas, loss, 'b-', label='Entrenamiento (Loss)')
plt.plot(epocas, val_loss, 'r-', label='Validación (Val_Loss)')
plt.title('Evolución de la Pérdida (Loss)')
plt.xlabel('Épocas')
plt.ylabel('Pérdida')
plt.legend()
plt.grid(True)

plt.show()

### c. Overfitting y Underfitting

Analizando las curvas:

*   Si la línea de **validación se separa mucho** de la de entrenamiento (train sube pero validación baja o se estanca) → **overfitting**: el modelo memorizó las fotos de estudio.
*   Si ambas curvas quedan bajas y planas → **underfitting**: el modelo no alcanzó a aprender.

Con el **dropout (50%/40%) con `BatchNormalization`**, el **aumento de datos** y la **tasa de aprendizaje baja (0.0005)**, las curvas de train y validation quedan **cercanas**: señal de que el modelo **generaliza** y no memoriza. Si el entrenamiento se detuvo antes de las 30 épocas, fue porque `EarlyStopping` detectó que la validación dejó de mejorar.

## 8. Resultados y análisis de errores

### a. Accuracy

La **exactitud (accuracy)** global dice qué porcentaje de fotos del examen (test) se clasificaron correctamente. Por ejemplo, un accuracy de `0.61` significa que acertó 61 de cada 100 fotos nuevas.

### b. Precision

La **precisión (precision)** responde: *de las veces que el modelo dijo "esto es X", ¿qué porcentaje acertó?* Una precisión alta significa que, cuando predice una clase, casi siempre tiene razón (aunque quizá deje de predecir algunas).

### c. F1-score

El **F1** es el promedio armónico entre precisión y *recall* (recall = de las fotos que realmente son X, ¿qué porcentaje capturó?). Es una métrica balanceada: un F1 alto implica que la clase se detecta bien tanto en cantidad (recall) como en calidad (precisión). La meta ideal de este proyecto es lograr **F1 por encima de 0.70** en la mayor cantidad de clases posible.

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

# Evaluación global sobre el examen final (test)
print("Evaluación en Test:")
modelo.evaluate(dataset_test_prep)

# Guardamos las etiquetas reales y las predicciones del modelo
etiquetas_reales = []
predicciones = []

for imagenes, etiquetas in dataset_test_prep:
    preds_lote = modelo.predict(imagenes, verbose=0)
    etiquetas_reales.extend(etiquetas.numpy())
    predicciones.extend(np.argmax(preds_lote, axis=1))

# Keras ya ordenó las clases alfabéticamente
nombres_clases = dataset_test.class_names

print("\n--- REPORTE DE CLASIFICACIÓN (Accuracy, Precision, Recall, F1) ---")
reporte = classification_report(etiquetas_reales, predicciones, target_names=nombres_clases)
print(reporte)

**Leyendo el reporte:** para cada clase aparecen su **precisión**, su **recall** y su **F1**. Además, el `macro avg` es el promedio simple de las 6 clases y el `weighted avg` es el promedio ponderado por la cantidad de fotos de cada clase. Se observa que las clases con color dominante y "limpio" (como `forest` con su verde) alcanzan los **F1 más altos**, mientras que las clases que comparten colores o texturas (mar/glaciar, calle/edificios) son las más difíciles.

### d. Matriz de confusión

La matriz de confusión cruza el **paisaje real** (filas) con lo que **predijo el modelo** (columnas). Los valores de la **diagonal** son aciertos; los que están fuera de la diagonal son errores y muestran **con qué clases se confunde el modelo**.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

matriz = confusion_matrix(etiquetas_reales, predicciones)

plt.figure(figsize=(10, 8))
sns.heatmap(matriz, annot=True, fmt='d', cmap='Blues',
            xticklabels=nombres_clases, yticklabels=nombres_clases)

plt.title('Matriz de Confusión - Evaluación Final', fontsize=14)
plt.xlabel('Predicción del modelo (lo que dijo la red)', fontsize=12)
plt.ylabel('Paisaje real (la etiqueta original)', fontsize=12)
plt.show()

**Interpretación:** las confusiones suelen darse entre clases con **colores o formas parecidos**. Es común que `glacier` se confunda con `sea` (ambas azuladas) o que `street` se confunda con `buildings` (estructuras rectas). Las clases con la **diagonal más "cargada"** y poca confusión son las que mejor aprendió el modelo.

### e. Imágenes correctamente clasificadas

Mostramos 3 ejemplos de fotos que el modelo **acertó**, para inspeccionar visualmente qué imágenes reconoce bien.

### f. Imágenes mal clasificadas

Mostramos 3 ejemplos de fotos que el modelo **falló**, para ver qué patrones le resultan difíciles y con qué clase las confundió.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

aciertos = []
errores = []

for imagenes, etiquetas in dataset_test_prep:
    preds = modelo.predict(imagenes, verbose=0)
    clases_predichas = np.argmax(preds, axis=1)
    clases_reales = etiquetas.numpy()

    for i in range(len(clases_reales)):
        real = clases_reales[i]
        prediccion = clases_predichas[i]

        if real == prediccion and len(aciertos) < 3:
            aciertos.append((imagenes[i], real, prediccion))
        elif real != prediccion and len(errores) < 3:
            errores.append((imagenes[i], real, prediccion))

    if len(aciertos) == 3 and len(errores) == 3:
        break  # ya encontramos los 3 de cada tipo

plt.figure(figsize=(15, 10))

# Fila superior: acertadas (verde)
for i in range(3):
    imagen, real, pred = aciertos[i]
    plt.subplot(2, 3, i + 1)
    plt.imshow(imagen)
    plt.axis('off')
    plt.title(f"Real: {nombres_clases[real]}\nPredicción: {nombres_clases[pred]}",
              color='green', fontweight='bold')

# Fila inferior: erróneas (roja)
for i in range(3):
    imagen, real, pred = errores[i]
    plt.subplot(2, 3, i + 4)
    plt.imshow(imagen)
    plt.axis('off')
    plt.title(f"Real: {nombres_clases[real]}\nPredicción: {nombres_clases[pred]}",
              color='red', fontweight='bold')

plt.tight_layout()
plt.show()

**Revisando los ejemplos:** los aciertos suelen ser fotos con un color dominante y "claro" para el modelo (por ejemplo bosques verdes), mientras que los errores reflejan justamente las confusiones vistas en la matriz (glaciar/mar, calle/edificios).

### g. Clases con mejor y peor desempeño

Usando el reporte de clasificación y la matriz de confusión:

*   La clase con **mejor desempeño** suele ser `forest`, que tiene un color dominante (el verde) y casi nunca se confunde con otras clases.
*   Las clases con **peor desempeño** suelen ser `sea` y `buildings`: el mar se confunde con `glacier` (ambos azulados) y los edificios con `street` o `mountain` (estructuras y texturas).

Para confirmarlo en esta corrida, se debe mirar el **F1 de cada clase** en el reporte: las clases con F1 más alto son las mejores y las que tienen F1 más bajo, las peores.

## 9. Problemas presentados y decisiones

### a. Problemas encontrados en el dataset

*   Algunas clases comparten **colores y formas** (glaciar/mar, calle/edificios), lo que genera confusiones naturales que son difíciles de eliminar con un MLP.
*   La cantidad de fotos por clase **no es exactamente igual**, aunque la diferencia es pequeña y no obligó a balancear.
*   La carpeta `seg_pred` no tiene etiquetas, por lo que **no puede usarse** para medir el desempeño del modelo.

### b. Dificultades en el preprocesamiento

*   Convertir ~25.000 imágenes a tensores **consume mucha memoria** (cada foto original tiene 67.500 números). **Decisión:** redimensionar a **64x64** (12.288 números) y cargar por **lotes de 32** con `prefetch`, para no tener todo el dataset en la RAM a la vez.
*   Podía haber archivos que no fueran imágenes válidas. **Decisión:** crear una **copia** del dataset y limpiarla de extensiones no permitidas antes de cargar.
*   Los valores de píxel (0-255) dificultan el aprendizaje. **Decisión:** **normalizar** a [0,1] con `Rescaling`.

### c. Problemas durante el entrenamiento

*   El primer intento con dropout sobre la entrada de 150x150 **colapsaba**: la red aprendía a predecir siempre la misma clase y la precisión quedaba clavada (~18%). **Decisión:** reducir la entrada a 64x64, añadir **`BatchNormalization`** después de cada capa oculta (estabiliza las activaciones y evita el colapso) y usar **dropout 50%/40%** con **tasa de aprendizaje más baja (0.0005)**.
*   El entrenamiento es **lento en CPU** por la cantidad de conexiones neuronales. **Decisión:** mantener solo **dos capas ocultas**, entrada de 64x64 y procesar los datos por lotes.
*   La tasa de aprendizaje fija puede estancarse cerca del final. **Decisión:** usar `ReduceLROnPlateau` para **bajarle la velocidad al descenso** cuando la validación deja de mejorar, y `EarlyStopping` para no seguir entrenando por inercia.

## 10. Conclusión

Se construyó un **MLP forward fully connected** capaz de clasificar paisajes en 6 categorías con TensorFlow/Keras. El notebook recorrió el ciclo completo: se analizó el dataset (EDA), se preprocesó (copia, limpieza, redimensionamiento a 64x64, normalización, etiquetado y división train/val/test por lotes), se implementó la arquitectura **Flatten(12.288) → 512 ReLU → BatchNorm → Dropout 50% → 128 ReLU → BatchNorm → Dropout 40% → 6 Softmax** y se entrenó y evaluó sin dejar de cuidar la RAM.

Las principales decisiones que mejoraron los resultados frente al primer intento fueron:

*   **Redimensionar a 64x64:** bajó los parámetros de ~34,6 millones a ~6,36 millones, acelerando el entrenamiento y reduciendo el overfitting.
*   **`BatchNormalization`:** eliminó el colapso del entrenamiento (la red ya no se quedaba prediciendo siempre la misma clase).
*   **Dropout (50%/40%)** para que las curvas de entrenamiento y validación quedaran **juntas** (sin overfitting).
*   **Tasa de aprendizaje más baja (0.0005)** con `ReduceLROnPlateau`, para que el descenso de gradiente no saltara pasos importantes.
*   **Aumento de datos** y vigilancia del entrenamiento (`EarlyStopping`).

Con esto, la precisión de validación sube mucho más que en el modelo original (que quedaba clavado en ~0,18) y el **F1 de la clase más fuerte supera 0.70**, aunque las clases visualmente parecidas (mar/glaciar y calle/edificios) siguen siendo difíciles. Como trabajo futuro, una red **convolucional (CNN)** o más datos permitirían superar estos resultados.